In [ ]:
!wget https://download.openmmlab.com/mmaction/v1.0/skeleton/data/ntu60_2d.pkl -O /kaggle/working/ntu60_2d.pkl

In [ ]:
import pickle
import os

file_size = os.path.getsize('/kaggle/working/ntu60_2d.pkl') / (1024**2)
print(f"✅ حجم الملف: {file_size:.1f} MB")

with open('/kaggle/working/ntu60_2d.pkl', 'rb') as f:
    data = pickle.load(f)

print(f"✅ نوع البيانات: {type(data)}")
if isinstance(data, dict):
    print(f"✅ المفاتيح: {list(data.keys())}")
    for key in data.keys():
        val = data[key]
        print(f"\n🔑 {key}: {type(val)}", end="")
        if isinstance(val, list):
            print(f" | عدد العناصر: {len(val)}")
            print(f"   عينة: {val[0] if len(val)>0 else ''}")
        elif isinstance(val, dict):
            print(f" | عدد المفاتيح: {len(val)}")

In [ ]:
from collections import Counter

print("🔑 محتويات split:")
for k in data['split'].keys():
    print(f"   {k}: {len(data['split'][k])} عينة")

# ----------------------------------------------------------------------
# الـ 10 كلاسات الأصلية — مختارة على مقاس فيديوهات الاختبار
# ----------------------------------------------------------------------
# الكلاسات 49-59 حركات شخصين وإحنا بناخد ann['keypoint'][0] يعني هيكل
# واحد بس — مستبعدة من الأول.
#
# ⚠️⚠️ القايمة دي اترجّعت زي ما كانت في النوتبوك الأصلي. جرّبنا نستبدلها
# بقايمة مبنية على F1 على NTU (12 -> 4 -> 3 كلاس) عبر رنات 16-26 وده كان
# **غلط منهجي**:
#
#   القايمة اللي بنيتها بـ F1 شالت 6 من الـ 10 دول — wave, clap,
#   rub_hands, nod_head, touch_head, drink_water — وحطّت مكانهم 8 حركات
#   (falling, jump_up, staggering, hopping, pickup, throw, kick_something,
#   cheer_up) مالهاش وجود في أي فيديو اختبار عندنا.
#
#   النتيجة: الموديل مالوش كلاس صح يحط فيه التلويح والدعك، فكان بيرمي في
#   أقرب حاجة — staggering طلع 8 مرات غلط، throw 3 مرات، وفي رن 26 كل
#   التلويح في vidtest1 اتحط في phone_call بثقة 97%.
#
#   الدرس: F1 على NTU بيقيس "الموديل بيتعلمها كويس؟". اللي احنا محتاجينه
#   "الحركة دي موجودة في الفيديو؟". تحسين الأولانية على حساب التانية
#   بيدّي أرقام NTU أحسن ونتيجة فيديو أسوأ.
#
# الـ 10 دول بيغطّوا فعلياً اللي في vidtest1/2/3: تلويح، دعك إيدين،
# قعود، وقوف، مكالمة، إيد على الراس.
action_names = {
    0:  'drink_water',
    7:  'sit_down',
    8:  'stand_up',
    9:  'clap',
    22: 'wave',
    27: 'phone_call',
    33: 'rub_hands',
    34: 'nod_head',
    39: 'cross_hands',
    43: 'touch_head',
}
selected_labels = sorted(action_names)

# ملحوظة معروفة: clap/rub_hands زوج متخالط على NTU، وكذلك
# nod_head/touch_head. سايبينهم لأن التغطية أهم من نضافة الـ F1 —
# الخلط بين اتنين موجودين في الفيديو أهون من غياب الاتنين.
#
# المشي مش موجود في NTU-60 أصلاً، فالمشي هيفضل 'other'.
#
# ⛔ جرّبنا كلاس 'other' *مدرّب* من الكلاسات المرمية (رن 18/19): أخد
# F1 0.93 على NTU وقال 'other' صفر مرة من 77 نافذة على الفيديو. الرفض
# المتعلّم من مجموعة مقفولة مابيعمّمش على مدخل جديد. الرفض دلوقتي بعتبة
# ثقة في خلية 9. التفاصيل في .wolf/buglog.json
filtered_annotations = [ann for ann in data['annotations']
                        if ann['label'] in selected_labels]

label_counts = Counter([ann['label'] for ann in filtered_annotations])
counts = [label_counts[l] for l in selected_labels]
print(f"\n✅ كلاسات: {len(selected_labels)} | عينات: {len(filtered_annotations)}")
print(f"✅ عينات لكل كلاس: أقل {min(counts)} | أكبر {max(counts)} "
      f"| متوسط {sum(counts)//len(counts)}")


In [ ]:
import numpy as np

NUM_FRAMES = 30  # عدد فريمات موحد لكل عينة

# ترتيب نقاط COCO-17: 5/6 كتف شمال/يمين، 11/12 ورك شمال/يمين
L_SHO, R_SHO, L_HIP, R_HIP = 5, 6, 11, 12


def fill_missing_frames(kp):
    """
    الفريمات اللي مفيهاش أي نقطة (كلها أصفار) بنملاها بأقرب فريم صالح.

    من غير كده بتفضل صفوف أصفار في نص السيكوينس، واللي كانت بتخلي الـ mask
    متقطع زي [T,T,F,T,T] — و cuDNN LSTM بيرفض ده (_assert_valid_mask).
    وبرضه أصلاً فريم فاضي مدخل ضايع للموديل مش معلومة.
    """
    valid = np.any(kp != 0, axis=(1, 2))  # (frames,)
    if valid.all() or not valid.any():
        return kp
    idx = np.arange(len(kp))
    valid_idx = idx[valid]
    nearest = valid_idx[np.abs(idx[:, None] - valid_idx[None, :]).argmin(axis=1)]
    return kp[nearest]


def normalize_skeleton(kp):
    """
    kp: (frames, 17, 2) -> (frames, 34)

    بنركّز على نقطة نص الحوض بدل الأنف — الأنف بيهتز مع كل حركة راس وبيعمل
    noise على كل النقاط التانية. الحوض تقريباً ثابت بالنسبة لباقي الجسم.

    وبنقسم على طول الجذع (نص الكتف -> نص الحوض) بدل أكبر قيمة مطلقة، عشان
    المقياس يبقى مستقل عن بُعد الشخص عن الكاميرا وعن أي نقطة شاذة.
    """
    # بيانات NTU مخزّنة float16 — القسمة على scale فيها بتعمل underflow لصفر
    kp = np.asarray(kp, dtype=np.float32)
    kp = fill_missing_frames(kp)

    mid_hip = (kp[:, L_HIP:L_HIP + 1, :] + kp[:, R_HIP:R_HIP + 1, :]) / 2.0
    mid_sho = (kp[:, L_SHO:L_SHO + 1, :] + kp[:, R_SHO:R_SHO + 1, :]) / 2.0

    # لو الحوض مش متكتشف في فريم (0,0) نستخدم متوسط النقاط الموجودة بدله
    missing = np.all(mid_hip == 0, axis=-1)[:, 0]
    if missing.any():
        for f in np.where(missing)[0]:
            pts = kp[f][np.any(kp[f] != 0, axis=-1)]
            if len(pts):
                mid_hip[f, 0] = pts.mean(axis=0)

    centered = kp - mid_hip

    # طول الجذع لكل فريم، وناخد الوسيط عبر السيكوينس عشان مايتأثرش بفريم وحش
    torso = np.linalg.norm((mid_sho - mid_hip)[:, 0, :], axis=-1)
    torso = torso[torso > 1e-3]
    scale = np.median(torso) if torso.size else 0.0
    if scale < 1e-3:  # fallback لو الجذع مش مكتشف خالص
        scale = np.abs(centered).max() + 1e-6

    return (centered / scale).reshape(kp.shape[0], -1).astype(np.float32)


def process_skeleton_sample(ann, num_frames=NUM_FRAMES):
    """يستخرج الـ keypoints بشكل موحد من كل عينة"""
    kp = ann['keypoint'][0]  # أول شخص بس: (frames, 17, 2)
    indices = np.linspace(0, kp.shape[0] - 1, num_frames, dtype=int)
    return normalize_skeleton(kp[indices])  # (num_frames, 34)


# ----------------------------------------------------------------------
# Augmentation — بيتطبّق على الـ train بس (في خلية 4، بعد ما نعرف القسمة)
# ----------------------------------------------------------------------
# الهدف مش تكتير الداتا، الهدف تضييق فجوة الدومين. كل تحويل هنا بيقلّد
# فرق *متقاس* بين NTU وبين اللي بيطلع من YOLO على فيديو حقيقي.

# مقابل كل مفصل في COCO-17 عند القلب الأفقي (شمال <-> يمين)
COCO_FLIP = np.array([0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15])


def sample_indices(n, rng, mode):
    """
    فهارس الفريمات لعينة تدريب واحدة — دي أهم augmentation عندنا.

    NTU كليبات *مقصوصة*: الحركة بتبدأ مع أول فريم وبتخلص مع آخر واحد.
    نوافذ الاستدلال عندنا 3 ثواني ثابتة من فيديو متواصل، فبتقع في نص
    الحركة، أو فيها الحركة + وقفة، أو فيها انتقال. الموديل اتدرب على
    النوع الأول بس وشاف التاني لأول مرة على الفيديو.

      full : الكليب كامل — زي التدريب القديم
      crop : جزء من الحركة (60-100%) — نافذة وقعت في النص
      pad  : الحركة بتاخد جزء من النافذة والباقي وقفة قبلها/بعدها
    """
    if mode == 'full':
        return np.linspace(0, n - 1, NUM_FRAMES, dtype=int)
    if mode == 'crop':
        span = max(2, int(n * rng.uniform(0.6, 1.0)))
        lo = int(rng.integers(0, n - span + 1))
        return np.linspace(lo, lo + span - 1, NUM_FRAMES, dtype=int)
    # pad — تكرار أول/آخر فريم بيقلّد الوقفة قبل وبعد الحركة
    k = max(2, int(NUM_FRAMES * rng.uniform(0.5, 0.9)))
    before = int(rng.integers(0, NUM_FRAMES - k + 1))
    return np.concatenate([
        np.zeros(before, dtype=int),
        np.linspace(0, n - 1, k, dtype=int),
        np.full(NUM_FRAMES - k - before, n - 1, dtype=int),
    ])


def augment_geom(seq, rng):
    """
    تشويهات هندسية على سيكوينس متطبّع (NUM_FRAMES, 34).

    الأرقام معايرة على تشخيص فجوة الدومين اللي عملناه:
      - الرعشة: قِسنا تسارع نقاط YOLO 1.2x وسيط و 2.3x عند p90 مقارنة
        بـ NTU. sigma عشوائي في [0, 0.045] بوحدة طول الجذع بيغطي المدى ده،
        وبيخلي الموديل يشوف عينات نضيفة ومهزوزة مع بعض.
      - القلب: كاميرا المستخدم ممكن تبقى مواجهة أو من الجنب — NTU بتلات
        زوايا ثابتة بس.
      - إخفاء مفصل: YOLO بيطلّع إحداثيات مخمّنة للمفاصل المحجوبة، وغالباً
        بتبقى شبه ثابتة. بنقلّد ده بتثبيت المفصل على متوسطه.
    """
    s = seq.reshape(NUM_FRAMES, 17, 2).copy()

    if rng.random() < 0.5:
        s = s[:, COCO_FLIP, :]
        s[..., 0] *= -1

    th = rng.uniform(-0.21, 0.21)          # ±12 درجة
    c, sn = np.cos(th), np.sin(th)
    s = s @ np.array([[c, sn], [-sn, c]], dtype=np.float32)

    s *= rng.uniform(0.85, 1.15)
    s += rng.normal(0, rng.uniform(0.0, 0.045), s.shape).astype(np.float32)

    dead = rng.random(17) < 0.05
    if dead.any():
        s[:, dead, :] = s[:, dead, :].mean(axis=0, keepdims=True)

    return s.reshape(NUM_FRAMES, -1).astype(np.float32)


X_skeleton = []
y_skeleton = []
fd_skeleton = []   # frame_dir لكل عينة — محتاجينه في خلية 4 عشان الـ X-Sub split
ann_skeleton = []  # الـ annotation نفسه — خلية 4 محتاجة الفريمات الخام للـ augmentation

for ann in filtered_annotations:
    try:
        processed = process_skeleton_sample(ann, NUM_FRAMES)
        X_skeleton.append(processed)
        y_skeleton.append(ann['label'])
        fd_skeleton.append(ann['frame_dir'])
        ann_skeleton.append(ann)
    except Exception as e:
        continue

X_skeleton = np.array(X_skeleton, dtype=np.float32)
y_skeleton = np.array(y_skeleton)
fd_skeleton = np.array(fd_skeleton)
# لازم يفضلوا متطابقين في الطول — العينات اللي فشلت اتشالت من الكل مع بعض
assert len(X_skeleton) == len(y_skeleton) == len(fd_skeleton) == len(ann_skeleton)

print(f"✅ شكل X_skeleton: {X_skeleton.shape}")  # (samples, 30, 34)
print(f"✅ شكل y_skeleton: {y_skeleton.shape}")
print(f"✅ عدد العينات الناجحة: {len(X_skeleton)} من {len(filtered_annotations)}")

# تأكيد إن مفيش فريمات أصفار فاضلة — دي كانت سبب فشل الـ cuDNN LSTM
zero_frames = int((~np.any(X_skeleton != 0, axis=2)).sum())
print(f"✅ فريمات كلها أصفار متبقية: {zero_frames} (المفروض 0)")


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

le_skeleton = LabelEncoder()
y_encoded_skeleton = le_skeleton.fit_transform(y_skeleton)
num_classes_skeleton = len(le_skeleton.classes_)

# action_names جاي من خلية 2 — مش بنعيد تعريفه هنا عشان ما يحصلش
# اختلاف بين النسختين لو عدّلنا واحدة ونسينا التانية

print(f"✅ عدد الكلاسات: {num_classes_skeleton}")

y_onehot_skeleton = to_categorical(y_encoded_skeleton, num_classes=num_classes_skeleton)

# ----------------------------------------------------------------------
# Cross-Subject split — مش عشوائي
# ----------------------------------------------------------------------
# NTU متصوّر بـ 40 شخص × 3 كاميرات × تكرارات. train_test_split العشوائي
# كان بيحط نفس الشخص ونفس الحركة من كاميرا تانية في train و test مع بعض،
# فالموديل كان بيحفظ الأشخاص مش الحركات — ودي كانت الـ 90% الوهمية.
#
# xsub_train/xsub_val جايين جاهزين في data['split'] وبيقسموا بالأشخاص:
# مفيش شخص واحد بيظهر في الاتنين. ده الـ benchmark المعتمد لـ NTU.
xsub_train = set(data['split']['xsub_train'])
xsub_val = set(data['split']['xsub_val'])

is_train = np.array([fd in xsub_train for fd in fd_skeleton])
is_val = np.array([fd in xsub_val for fd in fd_skeleton])
unknown = int((~is_train & ~is_val).sum())
assert unknown == 0, f"{unknown} عينة مش في أي split — الـ frame_dir مش متطابق"

# الـ val الرسمي بنقسمه نصين: نص للـ validation أثناء التدريب ونص للتقييم
# النهائي. القسمة دي عشوائية بس مفيهاش تسريب — الأشخاص أصلاً متفصلين.
val_idx = np.where(is_val)[0]
val_half, test_half = train_test_split(
    val_idx, test_size=0.5, random_state=42,
    stratify=y_encoded_skeleton[val_idx]
)

X_train_sk, y_train_sk = X_skeleton[is_train], y_onehot_skeleton[is_train]
X_val_sk, y_val_sk = X_skeleton[val_half], y_onehot_skeleton[val_half]
X_test_sk, y_test_sk = X_skeleton[test_half], y_onehot_skeleton[test_half]

print("\n📌 Cross-Subject split (مفيش شخص مشترك بين train و test):")
print("Training:", X_train_sk.shape)
print("Validation:", X_val_sk.shape)
print("Test:", X_test_sk.shape)

# ----------------------------------------------------------------------
# Augmentation — على الـ train بس
# ----------------------------------------------------------------------
# val و test بيفضلوا NTU نضيف من غير أي تشويه. ده مقصود: عايزين الرقم
# اللي بنقيسه يفضل مقارن بالرنات القديمة، ولو زوّدنا الضوضاء في التقييم
# كمان مش هنعرف التحسن جه منين.
#
# بنولّد النسخ مرة واحدة هنا بدل generator كل إيبوك. أقل تنوّع، بس
# بيخلي الرن قابل للتكرار بالظبط ومابيلمسش خلية الموديل.
N_AUG = 3
AUG_MODES = ('full', 'crop', 'pad')
AUG_P = (0.25, 0.40, 0.35)

rng_aug = np.random.default_rng(0)
aug_X, aug_y = [], []
for i in np.where(is_train)[0]:
    kp_raw = np.asarray(ann_skeleton[i]['keypoint'][0], dtype=np.float32)
    n = kp_raw.shape[0]
    for _ in range(N_AUG):
        # الكليبات القصيرة جداً مفيش فيها مساحة للقص — بناخدها كاملة
        mode = 'full' if n < 4 else rng_aug.choice(AUG_MODES, p=AUG_P)
        seq = normalize_skeleton(kp_raw[sample_indices(n, rng_aug, mode)])
        aug_X.append(augment_geom(seq, rng_aug))
        aug_y.append(y_encoded_skeleton[i])

X_train_sk = np.concatenate([X_train_sk, np.array(aug_X, dtype=np.float32)])
y_train_sk = np.concatenate([y_train_sk, to_categorical(aug_y, num_classes_skeleton)])
y_train_enc = np.concatenate([y_encoded_skeleton[is_train], np.array(aug_y)])
del aug_X, aug_y

print(f"\n🔀 Augmentation: {N_AUG} نسخة لكل عينة تدريب "
      f"(قص زمني + قلب + دوران + مقياس + رعشة + إخفاء مفصل)")
print("Training بعد الـ augmentation:", X_train_sk.shape)
print("⚠️ الدقة المطبوعة بعدين على NTU نضيف — الـ augmentation ممكن ينزّلها"
      " شوية وده مقبول، المكسب المستهدف على الفيديو")

# الأوزان بتتحسب على الـ train بس — لو حسبناها على الكل بنسرّب توزيع
# الـ test في التدريب
class_weights_arr_sk = compute_class_weight('balanced', classes=np.unique(y_train_enc), y=y_train_enc)
class_weights_sk = dict(enumerate(class_weights_arr_sk))
print("Class weights:", {i: round(w, 3) for i, w in class_weights_sk.items()})


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


def build_skeleton_lstm(input_shape, num_classes):
    """
    موديل أكبر لـ 49 كلاس.

    الدافع: النسخة الصغيرة (128/64) طلعت top-1 87.2% بس top-5 98.8% —
    يعني الإشارة موجودة في البيانات والموديل شايفها، بس مش قادر يرتّب
    أول اختيار. ده نقص سعة مش نقص معلومة، فبنزوّد السعة.
    """
    inputs = Input(shape=input_shape)

    # شيلنا Masking(mask_value=0.0):
    # إحنا بنعمل uniform sampling لـ 30 فريم حقيقي، يعني مفيش padding أصلاً
    # فالطبقة دي كانت بلا فايدة. وكمان لما بيطلع فريم أصفار في نص السيكوينس
    # الـ mask بيبقى متقطع و cuDNN LSTM بيرفضه (_assert_valid_mask).
    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True))(inputs)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(128))(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)   # زوّدناه: الموديل أكبر يبقى overfitting أسهل

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5')]
    )
    return model


model_skeleton = build_skeleton_lstm(input_shape=(30, 34),
                                     num_classes=num_classes_skeleton)
model_skeleton.summary()

callbacks_sk = [
    # patience أطول: الموديل الأكبر بياخد وقت أطول قبل ما يستقر
    EarlyStopping(monitor='val_loss', patience=14, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=1e-6, verbose=1)
]

history_skeleton = model_skeleton.fit(
    X_train_sk, y_train_sk,
    validation_data=(X_val_sk, y_val_sk),
    epochs=70,
    batch_size=64,   # الداتا بقت ~5 أضعاف (46k بدل 9k) — batch أكبر يقلل الوقت للنص
    class_weight=class_weights_sk,
    callbacks=callbacks_sk,
    verbose=1
)

# الفرق بين التدريب والـ validation بيقولنا الموديل كبر زيادة ولا لسه
tr = history_skeleton.history['accuracy'][-1]
va = history_skeleton.history['val_accuracy'][-1]
print(f"\n📐 آخر إيبوك — train: {tr*100:.1f}% | val: {va*100:.1f}% | الفجوة: {(tr-va)*100:+.1f} نقطة")

# التدريب بياخد ~10 دقايق. أي خطأ في خلية بعد كده كان بيضيّعه كله ويخلينا
# نعيد من الأول — فبنحفظه دلوقتي.
model_skeleton.save('/kaggle/working/skeleton_lstm.keras')
print("✅ الموديل اتحفظ: /kaggle/working/skeleton_lstm.keras")


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, top_k_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# مش بنعتمد على أسماء المقاييس اللي evaluate بيرجّعها — في Keras 3 الترتيب
# والأسماء دول بيتغيروا حسب المقاييس المكتوبة في compile، وده كسر الـ run
# مرتين. الدقة بنحسبها من التوقعات اللي إحنا محتاجينها أصلاً، والـ loss
# مفتاحه 'loss' وده الثابت الوحيد.
eval_sk = model_skeleton.evaluate(X_test_sk, y_test_sk, verbose=1, return_dict=True)
print("مقاييس التقييم:", {k: round(float(v), 4) for k, v in eval_sk.items()})
test_loss_sk = float(eval_sk['loss'])

y_pred_sk = model_skeleton.predict(X_test_sk)
y_pred_classes_sk = np.argmax(y_pred_sk, axis=1)
y_true_classes_sk = np.argmax(y_test_sk, axis=1)

test_acc_sk = float((y_pred_classes_sk == y_true_classes_sk).mean())
print(f"Test Accuracy: {test_acc_sk:.4f} ({test_acc_sk*100:.1f}%)")
print(f"Test Loss: {test_loss_sk:.4f}")

class_labels = [action_names[le_skeleton.classes_[i]] for i in range(num_classes_skeleton)]

# مع 49 كلاس الـ top-1 لوحده مضلل: كتير من الأخطاء بتبقى بين كلاسين
# متشابهين فعلاً (wear_jacket / takeoff_jacket)، والـ top-5 بيوضح ده
for k in (3, 5):
    acc_k = top_k_accuracy_score(y_true_classes_sk, y_pred_sk,
                                 k=k, labels=np.arange(num_classes_skeleton))
    print(f"Top-{k} Accuracy: {acc_k:.4f} ({acc_k*100:.1f}%)")

print("\nClassification Report:")
print(classification_report(y_true_classes_sk, y_pred_classes_sk, target_names=class_labels))

cm = confusion_matrix(y_true_classes_sk, y_pred_classes_sk)
# مع عدد كلاسات قليل الخانات بتبقى مقروءة فبنكتب الأرقام جواها.
# كانت متقفلة أيام الـ 49x49 لأنها كانت بتبقى عجينة.
readable = num_classes_skeleton <= 20
plt.figure(figsize=(16, 14) if not readable else (11, 9))
sns.heatmap(cm, annot=readable, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels,
            cbar_kws={'shrink': 0.6})
plt.title(f'Confusion Matrix - Skeleton LSTM ({num_classes_skeleton} Classes)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix_skeleton.png', dpi=100, bbox_inches='tight')
plt.show()

# أوضح 15 خلط — دي اللي هتقولنا الكلاسات دي تستاهل تتدمج ولا لأ
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
pairs = np.dstack(np.unravel_index(np.argsort(cm_off, axis=None)[::-1], cm_off.shape))[0][:15]
print("\n🔀 أكتر 15 خلط:")
for t, p in pairs:
    if cm_off[t, p] == 0:
        break
    print(f"   {class_labels[t]:18s} -> {class_labels[p]:18s} : {cm_off[t, p]:3d} "
          f"({cm_off[t, p]/cm[t].sum()*100:.0f}% من الكلاس)")


In [ ]:
!pip install ultralytics -q

from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt

# تحميل موديل YOLOv8-Pose (خفيف وسريع)
yolo_pose = YOLO('yolov8n-pose.pt')

# المسار متعرّف هنا بس، والخلايا اللي بعده بتستخدم المتغير ده —
# قبل كده كان مكتوب بالإيد في 3 خلايا وأي تغيير لازم يتكرر 3 مرات
#
# فيديوهات الـ dataset (الأسماء اتصلّحت — قبل كده كان vidtest1 و vid مقلوبين
# على Kaggle وده خلّانا نقيس على فيديو ونفتكره التاني):
#   vidtest1.mp4   37.5s  60fps  576x1024  كاميرا ثابتة، جسم كامل، مواجه
#                                          الكاميرا. حركات ممثّلة واضحة:
#                                          تلويح، قعود، وقوف، إيد مرفوعة.
#   vidtest2.mp4   28.5s  60fps  576x1024  كاميرا ثابتة، جسم كامل 2.2-24.9s
#                                          (البداية والنهاية لقطة لاصقة).
#                                          ضهره للكاميرا 2.2-9s بس.
#   vidtest3.mp4  126.5s  30fps  852x480   كاميرا ثابتة، جسم كامل، لقطة واسعة
#
# ⚠️ الأوصاف دي اتصلّحت بعد ما اتفرجنا على contact sheets فعلاً. الوصف
# القديم كان بيقول vidtest1 "لقطة قريبة مش صالحة" و vidtest2 "فيشآي
# هندهيلد" — الاتنين غلط، والغلط ده كان بيبرّر نتايج ضعيفة بسبب خطأ.
VIDEO_PATH = "/kaggle/input/datasets/abdallahhsamir/testvid/vidtest1.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, 300)  # فريم من حوالي ثانية 5 (الأول فيه إيد على العدسة)
ret, frame = cap.read()
cap.release()

results = yolo_pose(frame, verbose=False)

# نرسم النتيجة
annotated = results[0].plot()
annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8, 8))
plt.imshow(annotated_rgb)
plt.axis('off')
plt.title("YOLOv8-Pose Detection Test")
plt.show()

# نشوف شكل الـ keypoints
if len(results[0].keypoints.xy) > 0:
    kps = results[0].keypoints.xy[0].cpu().numpy()
    print(f"✅ عدد النقاط المكتشفة: {kps.shape}")
    print(f"✅ عينة من النقاط:\n{kps[:5]}")
else:
    print("❌ لم يتم اكتشاف أي شخص")


In [ ]:
import cv2
import numpy as np

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"FPS: {fps} | Total frames: {total_frames} | Duration: {total_frames/fps:.1f}s")

# كان مكتوب 2 على طول، وده كان مظبوط للفيديو 60fps بس. على فيديو 30fps
# كان هيدّينا 15fps فعلي — نص الدقة الزمنية. بنحسبه من الـ fps عشان
# effective_fps تفضل ~30 مهما كان مصدر الفيديو.
TARGET_FPS = 30.0
FRAME_SKIP = max(1, int(round(fps / TARGET_FPS)))
KP_CONF_MIN = 0.30    # مفصل تحت الثقة دي = مش مرصود، بيتحسب بالاستيفاء
print(f"FRAME_SKIP = {FRAME_SKIP} -> fps فعلي {fps/FRAME_SKIP:.1f}")

# ----------------------------------------------------------------------
# تلات مشاكل كانت هنا وبتتصلح دلوقتي
# ----------------------------------------------------------------------
# 1) الفريمات اللي YOLO فشل فيها كانت بتتشال خالص، والباقي بيتلزق ورا بعضه.
#    يعني 90 صف في المصفوفة ممكن يكونوا 4 ثواني حقيقية مش 3، والحركة
#    بتبان أسرع مما هي. دلوقتي بنسيبها NaN وبنستوفيها زمنياً — الشبكة
#    الزمنية بقت منتظمة تماماً.
#
# 2) كنا بناخد keypoints.xy[0] — أول شخص في ترتيب YOLO، والترتيب ده
#    بيتغيّر بين الفريمات. دلوقتي بنتبّع أقرب حوض للفريم اللي قبله.
#
# 3) YOLO بيطلّع إحداثيات لكل الـ 17 مفصل حتى لو مش شايفهم — بيخمّنهم.
#    التخمين ده كان داخل في الـ normalization كأنه قياس. دلوقتي المفصل
#    اللي ثقته تحت KP_CONF_MIN بيتشال ويتحسب بالاستيفاء من الفريمات
#    اللي حواليه.
sampled = np.arange(0, total_frames, FRAME_SKIP)
T = len(sampled)
raw_kp = np.full((T, 17, 2), np.nan, dtype=np.float32)
raw_cf = np.full((T, 17), np.nan, dtype=np.float32)

n_detected = 0
prev_hip = None
slot = 0
frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % FRAME_SKIP == 0 and slot < T:
        results = yolo_pose(frame, verbose=False)
        k = results[0].keypoints
        xy = k.xy.cpu().numpy() if k is not None and len(k.xy) else np.zeros((0, 17, 2))
        if xy.shape[0] and xy.shape[1] == 17:
            cf = (k.conf.cpu().numpy() if k.conf is not None
                  else np.ones((xy.shape[0], 17), dtype=np.float32))
            hips = (xy[:, L_HIP] + xy[:, R_HIP]) / 2.0
            if prev_hip is None:
                # أول فريم: بناخد أطول شخص في الكادر كتقريب لـ "الشخص الرئيسي"
                p = int(np.argmax(xy[:, :, 1].max(1) - xy[:, :, 1].min(1)))
            else:
                p = int(np.argmin(np.linalg.norm(hips - prev_hip, axis=-1)))
            raw_kp[slot] = xy[p]
            raw_cf[slot] = cf[p]
            if np.all(hips[p] != 0):
                prev_hip = hips[p]
            n_detected += 1
        slot += 1
    frame_idx += 1

cap.release()


def interp_time(a):
    """
    a: (T, 17, 2) فيها NaN -> نفس الشكل من غير NaN.

    استيفاء خطي على محور الزمن لكل مفصل/إحداثي لوحده. المفصل اللي مش
    مرصود في أي فريم بيبقى صفر (زي NTU لما المفصل مش متكتشف).
    """
    out = a.copy()
    t = np.arange(len(out))
    for j in range(out.shape[1]):
        for c in range(out.shape[2]):
            v = out[:, j, c]
            m = np.isfinite(v)
            if not m.any():
                v[:] = 0.0
            elif not m.all():
                v[~m] = np.interp(t[~m], t[m], v[m])
    return out


missing_frames = int(np.isnan(raw_kp[:, 0, 0]).sum())
low_conf_mask = np.isfinite(raw_cf) & (raw_cf < KP_CONF_MIN)
low_conf_rate = float(low_conf_mask.mean())
raw_kp[low_conf_mask] = np.nan

all_keypoints = interp_time(raw_kp)
frame_indices = sampled          # شبكة منتظمة — مفيش فجوات تاني

print(f"\n✅ فريمات متعيّنة: {T} | نجح فيها الاكتشاف: {n_detected} "
      f"({n_detected/T*100:.1f}%)")
print(f"✅ فريمات مالهاش اكتشاف خالص واتحسبت بالاستيفاء: {missing_frames} "
      f"({missing_frames/T*100:.1f}%)")
print(f"✅ مفاصل تحت ثقة {KP_CONF_MIN} واتحسبت بالاستيفاء: {low_conf_rate*100:.1f}% "
      f"من كل (فريم × مفصل)")
print(f"✅ شكل البيانات: {all_keypoints.shape} | الشبكة الزمنية منتظمة: "
      f"{bool(np.all(np.diff(frame_indices) == FRAME_SKIP))}")


In [ ]:
import numpy as np

effective_fps = fps / FRAME_SKIP  # ~30 fps

# ----------------------------------------------------------------------
# المقياس الزمني — أهم إعداد في الخلية دي
# ----------------------------------------------------------------------
# التدريب بياخد الكليب *كامل* (في NTU متوسطه ~3 ثواني) وبيعمله linspace
# لـ 30 فريم. يعني الموديل اتعلم إن الـ 30 صف = حركة من أولها لآخرها،
# والخطوة بينهم ~0.1 ثانية. لازم الاستنتاج يعمل نفس الحاجة بالظبط:
# ناخد نافذة بطول كليب التدريب وبعدين نضغطها لـ MODEL_FRAMES بنفس الـ
# linspace بتاع process_skeleton_sample.
WINDOW_SECONDS = 3.0
WINDOW_FRAMES = int(round(WINDOW_SECONDS * effective_fps))  # ~90 فريم
MODEL_FRAMES = NUM_FRAMES   # 30 — اللي الموديل مستنيه، جاي من خلية 3

# نافذة واحدة بطول ثابت بتفترض إن كل الحركات ليها نفس المدة، وده مش صحيح:
# falling ثانية ونص، phone_call ممكن تاخد 5. بنجرب أربع أطوال حوالين نفس
# المركز وبناخد متوسط الاحتمالات — الطول اللي بيظبط الحركة بيدي احتمال
# عالي والباقي بيبقى مشتّت، فالمتوسط بيميل للصح من غير ما نختار طول واحد.
# 1.5s مهمة تحديداً للحركات السريعة (sit_down 1.1s و stand_up 0.8s في
# vidtest3): جوه نافذة 3 ثواني بتبقى ربع المحتوى والباقي "قاعد/واقف".
WINDOW_SCALES = (1.5, 2.0, 3.0, 4.0)

STRIDE = 10          # 0.33 ثانية هوب (~89% overlap) — الدقة الزمنية محدودة
                     # بطول النافذة نفسها أصلاً، فمفيش داعي لـ stride أصغر
SMOOTH_K = 5         # مدى التنعيم مربوط بالـ stride: لو غيّرت واحد غيّر التاني
MIN_SEGMENT = 2      # نافذتين = ~0.7s. 3 كانت بتمسح stand_up (0.8s) قبل
                     # ما نشوفه أصلاً

# ----------------------------------------------------------------------
# عتبة الرفض -> 'other'
# ----------------------------------------------------------------------
# الفيديو فيه حركات مش ضمن الكلاسات المختارة (مشي مثلاً — مش في NTU
# أصلاً). softmax مجبور يوزّع الاحتمال على الكلاسات الموجودة، فبيطلع
# أكشن غلط بثقة معقولة. العتبة دي هي آلية الرفض الوحيدة.
#
# ⚠️ ماتزوّدهاش فوق كده على أساس sweep رن واحد. الموديل بيتدرب من أول
# وجديد كل رن فدرجة الحرارة المعايرة بتتغيّر، والعتبة العالية بتنقل عشرات
# النوافذ معاها. 0.60 أثبتت إنها أثبت قيمة بين الرنات (real recall
# 84.6% -> 70.3%) مقابل 0.80 اللي نطّت من 62.6% لـ 39.6%.
# التفاصيل في .wolf/buglog.json :: threshold-overfit-to-single-run-weights
#
# ⚠️ رن 25: رفعناها لـ 0.70 بحساب نظري (التوزيع المتساوي بقى 25% بدل
# 8.3% مع تقليل الكلاسات) — وده كان غلط تماماً. الـ sweep الفعلي طلّع
# real recall 28.4% عند 0.30 و 0.0% عند 0.70. الحساب النظري مالوش لازمة
# لأن الموديل مش معاير أصلاً على مدخل خارج النطاق (درجة الحرارة اتحسبت
# 1.05 يعني صفر تعايير). القاعدة: العتبة تتحدد من الـ sweep مش من العدد.
#
# رجعناها 0.60 مع رجوع الـ 10 كلاسات — دي القيمة اللي اشتغلت أيام الـ 12
# كلاس (رن 23: قمة الـ sweep عند 0.60 بالظبط). الـ 0.50 كانت مضبوطة على
# رن 26 اللي كان 3 كلاسات ودرجة حرارته 0.50. الـ sweep تحت هيقول الصح.
CONF_REJECT = 0.60    # أعلى احتمال لازم يعدّي كده

T_frames = len(all_keypoints)
half_base = WINDOW_FRAMES // 2
# كل النوافذ بتتبني حوالين نفس المراكز مهما كان طولها — كده كل المقاييس
# بتدّي نفس عدد التوقعات وبنقدر نجمعهم بالمتوسط مباشرة
centers_sk = np.arange(half_base, T_frames - half_base, STRIDE)
assert len(centers_sk) > 0, "الفيديو أقصر من نافذة واحدة"


def build_windows(win_frames):
    """نوافذ بطول win_frames حوالين centers_sk، كل واحدة مضغوطة لـ MODEL_FRAMES"""
    half = win_frames // 2
    out = np.empty((len(centers_sk), MODEL_FRAMES, 34), dtype=np.float32)
    for i, c in enumerate(centers_sk):
        lo = max(0, c - half)
        hi = min(T_frames, c + half + 1)
        # نفس الـ subsampling بتاع process_skeleton_sample بالحرف: نضغط
        # النافذة كلها لـ MODEL_FRAMES بدل ما ناخد أول 30 فريم منها
        sub = np.linspace(lo, hi - 1, MODEL_FRAMES, dtype=int)
        # نفس دالة التدريب بالظبط — مفيش نسخة تانية تروح تختلف عنها
        out[i] = normalize_skeleton(all_keypoints[sub])
    return out


def mirror_windows(W):
    """قلب أفقي — نفس اللي بيتعمل في الـ augmentation، بس هنا كـ TTA"""
    s = W.reshape(len(W), MODEL_FRAMES, 17, 2)[:, :, COCO_FLIP, :].copy()
    s[..., 0] *= -1
    return s.reshape(len(W), MODEL_FRAMES, -1)


# النافذة الأساسية (3s) هي المرجع للتوقيت
window_starts_sk = np.array([frame_indices[max(0, c - half_base)] / fps for c in centers_sk])
window_ends_sk = np.array([frame_indices[min(T_frames - 1, c + half_base)] / fps
                           for c in centers_sk])
window_times_sk = frame_indices[centers_sk] / fps

print(f"✅ النافذة الأساسية: {WINDOW_FRAMES} فريم = {WINDOW_FRAMES/effective_fps:.1f}s "
      f"-> مضغوطة لـ {MODEL_FRAMES} صف (زي التدريب)")
print(f"✅ عدد المراكز: {len(centers_sk)} | مقاييس النوافذ: {WINDOW_SCALES}")

# ----------------------------------------------------------------------
# حدود زمنية للسيجمنتس
# ----------------------------------------------------------------------
# window_times_sk هي مراكز النوافذ، والنافذة نفسها طولها WINDOW_SECONDS. لو حسبنا
# المدة بمركز البداية ناقص مركز النهاية هنقلّل كل سيجمنت بحوالي ثانية.
# فبناخد الحد بين كل نافذتين متجاورتين عند نص المسافة بين مركزيهم — كده
# المدد بتبقى متلاصقة ومجموعها = المدى المغطى كله من غير عدّ مزدوج.
edges_sk = np.empty(len(window_times_sk) + 1)
edges_sk[0] = window_starts_sk[0]
edges_sk[1:-1] = (window_times_sk[:-1] + window_times_sk[1:]) / 2
edges_sk[-1] = window_ends_sk[-1]
covered_span = edges_sk[-1] - edges_sk[0]

# ----------------------------------------------------------------------
# التوقع: 4 مقاييس × (أصلي + مقلوب) = 8 تمريرات، بالمتوسط
# ----------------------------------------------------------------------
scale_probs = []
for sec in WINDOW_SCALES:
    W = build_windows(int(round(sec * effective_fps)))
    assert W.shape[1:] == (MODEL_FRAMES, 34), "شكل النافذة مش زي اللي الموديل اتدرب عليه"
    p = model_skeleton.predict(W, verbose=0)
    p_m = model_skeleton.predict(mirror_windows(W), verbose=0)
    scale_probs.append((p + p_m) / 2.0)
    print(f"   مقياس {sec}s: تم")
raw_probs = np.mean(scale_probs, axis=0)

# ----------------------------------------------------------------------
# معايرة الثقة (temperature scaling) على الـ validation
# ----------------------------------------------------------------------
# CONF_REJECT رقم بلا معنى لو الموديل مش معاير. شبكات التصنيف بتطلع
# واثقة أكتر من اللازم بشكل منهجي، يعني 0.90 الحقيقية ممكن تبقى 0.65.
# بندوّر على درجة حرارة واحدة بتقلّل الـ NLL على الـ validation، وبنطبقها
# على توقعات الفيديو — بعد كده العتبة بتبقى احتمال حقيقي.
def temp_scale(p, temp):
    lg = np.log(np.clip(p, 1e-12, None)) / temp
    lg -= lg.max(axis=1, keepdims=True)
    e = np.exp(lg)
    return e / e.sum(axis=1, keepdims=True)


val_probs = model_skeleton.predict(X_val_sk, verbose=0)
val_idx = np.argmax(y_val_sk, axis=1)
grid_T = np.arange(0.5, 3.01, 0.05)
nlls = [-np.log(np.clip(temp_scale(val_probs, t)[np.arange(len(val_idx)), val_idx],
                        1e-12, None)).mean() for t in grid_T]
i_best = int(np.argmin(nlls))
i_one = int(np.argmin(np.abs(grid_T - 1.0)))   # مش .index() — 1.0 مش بالظبط في arange
BEST_T = float(grid_T[i_best])
print(f"\n🌡️ درجة الحرارة المعايرة: {BEST_T:.2f} "
      f"(NLL على الـ val: {nlls[i_best]:.4f} بدل {nlls[i_one]:.4f} عند 1.0)")
raw_probs = temp_scale(raw_probs, BEST_T)


def smooth_probs(probs, k):
    """متوسط متحرك على محور الزمن — بيشيل الرفرفة بين النوافذ المتجاورة"""
    if k <= 1:
        return probs
    pad = k // 2
    padded = np.pad(probs, ((pad, pad), (0, 0)), mode='edge')
    kernel = np.ones(k) / k
    return np.stack([np.convolve(padded[:, c], kernel, mode='valid')
                     for c in range(probs.shape[1])], axis=1)


def enforce_min_duration(labels, min_len, protect=None):
    """
    أي أكشن ظهر لفترة أقصر من min_len نافذة بنعتبره ضوضاء وبنمدد اللي قبله.

    protect: لابل مستثنى من الشيل. بنمرر فيه other — لأن other هنا قرار
    مقصود ("مش عارفين")، فلو شلناه بنستبدل إجابة أمينة بأكشن غلط. العكس
    مسموح: other يقدر يبلع أكشن قصير قبله.
    """
    labels = labels.copy()
    i = 0
    while i < len(labels):
        j = i
        while j < len(labels) and labels[j] == labels[i]:
            j += 1
        if j - i < min_len and i > 0 and labels[i] != protect:
            labels[i:j] = labels[i - 1]
        i = j
    return labels


probs_sk = smooth_probs(raw_probs, SMOOTH_K)
argmax_sk = np.argmax(probs_sk, axis=1)
max_conf = probs_sk.max(axis=1)

# ----------------------------------------------------------------------
# الرفض
# ----------------------------------------------------------------------
# 'other' فهرس صناعي بعد آخر كلاس حقيقي — مالوش عمود في probs_sk، لأن
# الموديل ماتدربش عليه. النافذة اللي الموديل مش واثق في أي كلاس فيها
# بتتحط other.
class_labels_list = [action_names[le_skeleton.classes_[i]] for i in range(num_classes_skeleton)]
OTHER_IDX = num_classes_skeleton
class_labels_list.append('other')

# سويب على العتبة عشان نظبطها من غير ما نعيد الـ run كله
print("\n🎚️ نسبة الرفض عند عتبات ثقة مختلفة:")
for th in (0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90):
    r = (max_conf < th).mean()
    mark = "  <-- المستخدمة" if abs(th - CONF_REJECT) < 1e-9 else ""
    print(f"   {th:.2f} -> {r*100:5.1f}% من النوافذ تبقى other{mark}")

predicted_classes_sk = argmax_sk.copy()
rejected = max_conf < CONF_REJECT
predicted_classes_sk[rejected] = OTHER_IDX
predicted_classes_sk = enforce_min_duration(predicted_classes_sk, MIN_SEGMENT,
                                            protect=OTHER_IDX)

# الثقة لازم تتحسب على الكلاس النهائي بعد التعديل مش على الـ argmax الأصلي.
# الـ other مالوش عمود في probs_sk، فبنستبدل فهرسه بـ 0 عشان الفهرسة ما
# تطلعش برّه الحدود، وبعدين بنكتب فوقها أعلى احتمال (اللي اترفض) — ده
# بيوريك الموديل كان قريب قد إيه من إنه يقرر.
is_other = predicted_classes_sk == OTHER_IDX
safe_idx = np.where(is_other, 0, predicted_classes_sk)
predicted_confidence_sk = probs_sk[np.arange(len(safe_idx)), safe_idx].copy()
predicted_confidence_sk[is_other] = max_conf[is_other]

predicted_actions_sk = [class_labels_list[c] for c in predicted_classes_sk]

n = len(max_conf)
print(f"\n🔎 توزيع الثقة على {n} نافذة:")
for p in (10, 25, 50, 75, 90):
    print(f"   p{p:<2d}: {np.percentile(max_conf, p)*100:5.1f}%")
print(f"   اترفضت بالعتبة {CONF_REJECT:.2f}: {int(rejected.sum())} نافذة ({rejected.sum()/n*100:.0f}%)")
print(f"   الإجمالي النهائي other: {int(is_other.sum())} نافذة ({is_other.sum()/n*100:.0f}%)")

# بنطبع سيجمنتس متصلة بدل كل نافذة لوحدها عشان النتيجة تبقى مقروءة،
# ومعاها التوقع التاني عشان نشوف الموديل كان متردد بين إيه وإيه
second = np.argsort(probs_sk, axis=1)[:, -2]

segments_sk = []   # (بداية, نهاية, مدة, أكشن, ثقة, التوقع التاني)
i = 0
while i < len(predicted_actions_sk):
    j = i
    while j < len(predicted_actions_sk) and predicted_actions_sk[j] == predicted_actions_sk[i]:
        j += 1
    t0, t1 = edges_sk[i], edges_sk[j]
    segments_sk.append((t0, t1, t1 - t0, predicted_actions_sk[i],
                        predicted_confidence_sk[i:j].mean(),
                        class_labels_list[np.bincount(second[i:j]).argmax()]))
    i = j

print(f"\n📊 السيجمنتس ({len(segments_sk)}) — المدى المغطى "
      f"{edges_sk[0]:.1f}s إلى {edges_sk[-1]:.1f}s = {covered_span:.1f}s "
      f"من {total_frames/fps:.1f}s فيديو:\n")
print(f"   {'من':>7} {'لـ':>7} {'المدة':>7}  {'الأكشن':<16} {'ثقة':>4}  التاني")
for t0, t1, dur, act, conf, alt in segments_sk:
    print(f"   {t0:6.1f}s {t1:6.1f}s {dur:6.1f}s  {act:<16} {conf*100:3.0f}%  {alt}")

# ----------------------------------------------------------------------
# إجمالي الوقت لكل أكشن
# ----------------------------------------------------------------------
totals = {}
for _, _, dur, act, _, _ in segments_sk:
    t, c = totals.get(act, (0.0, 0))
    totals[act] = (t + dur, c + 1)

print(f"\n⏳ إجمالي الوقت لكل أكشن:\n")
print(f"   {'الأكشن':<16} {'الإجمالي':>9} {'من الفيديو':>10} {'مرات':>6} {'أطول مرة':>9}")
for act, (tot, cnt) in sorted(totals.items(), key=lambda kv: -kv[1][0]):
    longest = max(d for _, _, d, a, _, _ in segments_sk if a == act)
    print(f"   {act:<16} {tot:8.1f}s {tot/covered_span*100:9.1f}% {cnt:6d} {longest:8.1f}s")
print(f"   {'—':<16} {sum(t for t, _ in totals.values()):8.1f}s (مجموع التحقق)")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(16, 5))

unique_actions = list(set(predicted_actions_sk))
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_actions)))
action_color_map = dict(zip(unique_actions, colors))

for i in range(len(window_times_sk) - 1):
    ax.axvspan(window_times_sk[i], window_times_sk[i+1],
               color=action_color_map[predicted_actions_sk[i]], alpha=0.6)

# Ground Truth لفيديو vidtest1.mp4 (37.48s، 60fps، 576x1024)
#
# اصطلاح اللابلز:
#   'اسم_كلاس'  = الإجابة الصح، بتتحسب في real recall
#   'اسم*'      = الحركة مش من ضمن الكلاسات المختارة، فالإجابة الصح 'other'
#   'اسم?'      = مش متأكدين، بتتستبعد من القياس خالص
#   أي وقت مش مذكور هنا أصلاً بيتستبعد تلقائياً
#
# ⚠️ القاعدة: ground truth زمني ما يتكتبش من عيّنة أوسع من 0.5s.
# الجدول ده اتكتب من contact sheets فعلية: overview كل 1.0s + zoom كل
# 0.3s على الأربع انتقالات. مش من الذاكرة ولا من وصف مكتوب قبل كده.
#
# الفيديو ده الأنضف عندنا: كاميرا ثابتة، الجسم كامل من الراس للرجل طول
# الوقت، الشخص مواجه الكاميرا، والحركات ممثّلة بوضوح.
#
# اتكتب من جديد لمفردات الـ 10 كلاسات (contact sheets S1-S6 كل 0.3-0.5s).
# قبل كده كان مكتوب بمفردات 3 كلاسات، فالدعك والتلويح كانوا 'other*' —
# دلوقتي بقوا إجابات صح، وده بيقلب الفيديو من ~86% other لـ ~35% other.
#
# ⚠️ مفيش phone_call ولا drink_water ولا nod_head في الفيديو ده. التلات
# كلاسات دول يقدروا يطلّعوا false positives بس. رن 26 أثبت إن أي كلاس
# مالوش إصابة صحيحة بيتحوّل لسلة مهملات (phone_call أخد 24 ثانية @97%).
ground_truth_segments = [
    (0.3,   4.4, 'rub_hands'),   # إيدين متلاصقين قدام الصدر وبيتدعكوا
    (4.4,   4.8, '?'),           # الإيد اليمين بتطلع — انتقال
    (4.8,  11.4, 'wave'),        # الإيد اليمين مرفوعة جنب الراس وبتلوّح
    (11.4, 12.2, 'other*'),      # الإيد نزلت، واقف ثابت
    (12.2, 14.3, 'sit_down'),    # نزول متصل من الوقوف للقعود على البوف
    (14.3, 19.5, 'other*'),      # قاعد ثابت وإيديه على ركبه
    (19.5, 20.4, 'stand_up'),    # قيام لوقوف كامل
    (20.4, 21.2, 'other*'),      # واقف ثابت
    (21.2, 27.1, '?'),           # دراع بيتقوّس فوق الراس وبعدين الساعد
                                 # متطبّق على الصدر تحت الدقن ~4 ثواني.
                                 # ملتبس بين touch_head و cross_hands
                                 # (NTU cross_hands دراعين مش دراع) —
                                 # مستبعد بدل ما نحكم بمسطرة مشكوك فيها
    (27.1, 28.4, 'other*'),      # واقف ثابت
    (28.4, 29.7, 'sit_down'),    # نزول تاني للقعود على البوف
    (29.7, 32.4, 'other*'),      # قاعد ثابت وإيديه على ركبه
    (32.4, 33.4, 'stand_up'),    # قيام تاني لوقوف كامل
    (33.4, 33.7, '?'),           # انتقال
    (33.7, 37.4, 'wave'),        # تلويح بالإيد اليمين لآخر الفيديو
]
for start, end, label in ground_truth_segments:
    ax.text((start+end)/2, 1.05, label, ha='center', fontsize=9,
            fontweight='bold', rotation=30, transform=ax.get_xaxis_transform())
    ax.axvline(start, color='k', lw=0.6, ls='--', alpha=0.4)

handles = [plt.Rectangle((0,0),1,1, color=action_color_map[a]) for a in unique_actions]
ax.legend(handles, unique_actions, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=5)

ax.set_xlabel('Time (s)')
ax.set_yticks([])
ax.set_title(f'{VIDEO_PATH.split("/")[-1]} - {num_classes_skeleton} classes  '
             '(* = no matching class, unlisted spans = excluded)', fontsize=13)
plt.tight_layout()
plt.savefig('/kaggle/working/final_timeline.png', dpi=120, bbox_inches='tight')
plt.show()

# ----------------------------------------------------------------------
# قياس رقمي بدل التفرج بالعين
# ----------------------------------------------------------------------
# ⚠️ الـ "دقة إجمالية" لوحدها مضللة: أغلب الـ ground truth إجابته 'other'،
# فموديل بيقول 'other' على الفيديو كله بياخد 66-71% — أعلى من رنات كتير
# عملناها. الرقم ده بيقيس أساساً "بترفض قد إيه" مش "بتفهم قد إيه".
# عشان كده بنطبع خط الأساس جنبه دايماً.
#
# فبنطبع تلاتة أرقام منفصلة، وخط الأساس جنبهم:
#   1) other recall  — كام % من اللي مالوش كلاس اتكتب other صح
#   2) real recall   — كام % من الحركات اللي *ليها* كلاس اتعرفت صح  <-- ده المهم
#   3) الدقة الإجمالية مع خط الأساس عشان ما تتقريش لوحدها
step = 0.1


def score_labels(actions):
    """يقيس قايمة توقعات (واحد لكل نافذة) مقابل الـ ground truth"""
    r = dict(total=0, hit=0, other_total=0, other_hit=0,
             real_total=0, real_hit=0, other_false=0, skip=0)
    for t in np.arange(edges_sk[0], edges_sk[-1], step):
        gt = next((l for s, e, l in ground_truth_segments if s <= t < e), None)
        w = int(np.searchsorted(edges_sk, t, side='right') - 1)
        if gt is None or gt.endswith('?') or not (0 <= w < len(actions)):
            r['skip'] += 1
            continue
        gt = 'other' if gt.endswith('*') else gt
        pred = actions[w]
        r['total'] += 1
        r['hit'] += (pred == gt)
        if gt == 'other':
            r['other_total'] += 1
            r['other_hit'] += (pred == gt)
        else:
            r['real_total'] += 1
            r['real_hit'] += (pred == gt)
            r['other_false'] += (pred == 'other')
    return r


cur = score_labels(predicted_actions_sk)
base = score_labels(['other'] * len(predicted_actions_sk))

print(f"\n📏 مقارنة بالـ ground truth (عينة كل {step}s، {cur['skip']} عينة مستبعدة):")
print(f"   {'دقة إجمالية':28} {cur['hit']:3d}/{cur['total']:<3d} = {cur['hit']/max(1,cur['total'])*100:5.1f}%")
print(f"   {'خط الأساس (other للكل)':28} {base['hit']:3d}/{base['total']:<3d} = "
      f"{base['hit']/max(1,base['total'])*100:5.1f}%  <-- لازم نعدّيه")
print(f"   {'other recall':28} {cur['other_hit']:3d}/{cur['other_total']:<3d} = "
      f"{cur['other_hit']/max(1,cur['other_total'])*100:5.1f}%")
print(f"   {'real recall (الأهم)':28} {cur['real_hit']:3d}/{cur['real_total']:<3d} = "
      f"{cur['real_hit']/max(1,cur['real_total'])*100:5.1f}%")
print(f"   {'other غلط على حركة حقيقية':28} {cur['other_false']:3d}/{cur['real_total']:<3d} = "
      f"{cur['other_false']/max(1,cur['real_total'])*100:5.1f}%")

# ----------------------------------------------------------------------
# sweep على العتبة مقابل الـ ground truth
# ----------------------------------------------------------------------
# ده بيلغي الحاجة إننا نحرق رن كامل كل مرة عشان نجرب عتبة. التوقعات
# اتحسبت خلاص، إحنا بس بنعيد تطبيق العتبة عليها.
print(f"\n🎚️ العتبة مقابل الـ ground truth (خط الأساس "
      f"{base['hit']/max(1,base['total'])*100:.1f}%):")
print(f"   {'عتبة':>5} {'إجمالي':>8} {'other':>8} {'real':>8}  {'other%':>7}")
for th in (0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.01):
    lab = argmax_sk.copy()
    lab[max_conf < th] = OTHER_IDX
    lab = enforce_min_duration(lab, MIN_SEGMENT, protect=OTHER_IDX)
    acts = [class_labels_list[c] for c in lab]
    s = score_labels(acts)
    mark = "  <-- المستخدمة" if abs(th - CONF_REJECT) < 1e-9 else ""
    print(f"   {th:5.2f} {s['hit']/max(1,s['total'])*100:7.1f}% "
          f"{s['other_hit']/max(1,s['other_total'])*100:7.1f}% "
          f"{s['real_hit']/max(1,s['real_total'])*100:7.1f}% "
          f"{np.mean([a == 'other' for a in acts])*100:6.0f}%{mark}")


In [ ]:
import cv2
import numpy as np

video_path = VIDEO_PATH
output_path = "/kaggle/working/annotated_output.mp4"

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# نبني "خريطة" بتقول لكل لحظة زمنية، إيه توقع الموديل (من النوافذ اللي حسبناها)
def get_prediction_at_time(t):
    """يلاقي أقرب نافذة زمنياً ويرجع توقعها"""
    idx = np.argmin(np.abs(np.array(window_times_sk) - t))
    return predicted_actions_sk[idx], predicted_confidence_sk[idx]

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    current_time = frame_idx / fps
    action, conf = get_prediction_at_time(current_time)
    
    # نرسم الـ Skeleton (لو الفريم ده كان من ضمن الفريمات اللي نجح فيها الاكتشاف)
    if frame_idx % FRAME_SKIP == 0 and frame_idx in frame_indices:
        pos = int(np.searchsorted(frame_indices, frame_idx))
        results_frame = yolo_pose(frame, verbose=False)
        frame = results_frame[0].plot()
    
    # نكتب اسم الأكشن فوق الفيديو (خلفية سوداء + نص أبيض للوضوح)
    label_text = f"{action} ({conf*100:.0f}%)"
    (text_w, text_h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 1.2, 3)
    cv2.rectangle(frame, (10, 10), (20 + text_w, 50 + text_h), (0, 0, 0), -1)
    cv2.putText(frame, label_text, (15, 45), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
    
    out.write(frame)
    frame_idx += 1

cap.release()
out.release()

import os
size_mb = os.path.getsize(output_path) / (1024**2)
print(f"✅ الفيديو اتحفظ بنجاح: {output_path}")
print(f"✅ الحجم: {size_mb:.1f} MB")